# 📘 Deploy Application Tools

> **Applicable Environment**: Kubernetes Pod (Ubuntu base image)
> **Purpose**: Install commonly used application tools needed for development and debugging, including opencode (AI programming assistant) and pbcopy (clipboard transfer tool).

## 1. Competition Introduction

This is an AMD Radeon-hackathon-2026-07 competition project.
- **Runtime Constraints**: No Docker/Podman inside the Pod, data persistence via PVC mounts, PostgreSQL managed by custom scripts
- **Key Paths**: The persistent directory `/workspace/persistent` is mapped to `/data` for unified service access

## 2. Install opencode

Download and install the opencode AI programming assistant to `/data/app`, and write its path to `~/.profile` for global use.

```python
%%bash
#!/bin/bash
set -euo pipefail

APP_DIR="/data/app"
OPENCODE_VERSION="v1.18.9"
TARBALL="opencode-linux-x64.tar.gz"
DOWNLOAD_URL="https://github.com/anomalyco/opencode/releases/download/${OPENCODE_VERSION}/${TARBALL}"
TMP_TAR="/tmp/${TARBALL}"

# 1. Create app directory (with error handling)
mkdir -p "$APP_DIR" || { echo "❌ Failed to create $APP_DIR, check permissions"; exit 1; }
echo "📁 App Directory: $APP_DIR"

# 2. Check if already installed
if [ -x "$APP_DIR/opencode" ]; then
    echo "✅ opencode installed: $($APP_DIR/opencode --version 2>/dev/null || echo 'Ready')"
    exit 0
fi

# 3. Determine download tool (curl or wget)
DOWNLOAD_CMD=""
if command -v curl &>/dev/null; then
    DOWNLOAD_CMD="curl -L --retry 3 --connect-timeout 10 -o \"$TMP_TAR\" \"$DOWNLOAD_URL\""
elif command -v wget &>/dev/null; then
    DOWNLOAD_CMD="wget --no-check-certificate --tries=3 --timeout=10 -O \"$TMP_TAR\" \"$DOWNLOAD_URL\""
else
    echo "❌ Neither curl nor wget found. Please install one of them."
    exit 1
fi

# 4. Download
echo "🔧 Downloading opencode ${OPENCODE_VERSION} using ${DOWNLOAD_CMD%% *}..."
eval "$DOWNLOAD_CMD" || { echo "❌ Download failed"; exit 1; }

# 5. Check file size (ensure download succeeded)
if [ ! -s "$TMP_TAR" ]; then
    echo "❌ Downloaded file is empty or missing"
    exit 1
fi
echo "✅ Download complete ($(du -h $TMP_TAR | cut -f1))"

# 6. Extract
echo "📦 Extracting..."
tar -xzf "$TMP_TAR" -C "$APP_DIR" || { echo "❌ Extraction failed"; exit 1; }
rm -f "$TMP_TAR"
echo "✅ Installation complete"

# 7. Add to ~/.profile (if not already present)
PROFILE_LINE='export PATH="$PATH:/data/app"'
if ! grep -qF "/data/app" "$HOME/.profile" 2>/dev/null; then
    echo "$PROFILE_LINE" >> "$HOME/.profile"
    echo "✅ /data/app added to ~/.profile"
else
    echo "✅ /data/app already in ~/.profile"
fi

# 8. Verify installation
if [ -x "$APP_DIR/opencode" ]; then
    echo ""
    echo "📊 Verification results:"
    "$APP_DIR/opencode" --version || echo "opencode is ready"
    ls -la "$APP_DIR"
    echo ""
    echo "💡 To use opencode immediately, run: source ~/.profile"
else
    echo "❌ opencode installation failed"
    exit 1
fi
```

## 3. Configure pbcopy

Install the `pbcopy` clipboard transfer tool (macOS-compatible style), used to copy content to the local clipboard via terminal OSC 52 sequence.

```python
%%bash
#!/bin/bash
set -euo pipefail

TARGET="/usr/local/bin/pbcopy"

# 1. Check if we have write permission to /usr/local/bin
if [ ! -w "$(dirname $TARGET)" ] && [ "$(id -u)" -ne 0 ]; then
    echo "❌ No write permission to /usr/local/bin and not root. Please run as root or use sudo."
    exit 1
fi

# 2. Write script
cat > "$TARGET" << 'EOF'
#!/bin/sh
# pbcopy - copy to clipboard via OSC 52
if [ -t 0 ]; then
    echo "Usage: echo 'text' | pbcopy" >&2
    exit 1
fi
printf '\033]52;c;%s\a' "$(base64 | tr -d '\n')"
EOF

# 3. Make executable
chmod +x "$TARGET"

# 4. Verify
if [ -x "$TARGET" ]; then
    echo "✅ pbcopy installed successfully: $TARGET"
    echo "--- Script content ---"
    cat "$TARGET"
    echo ""
    echo "💡 Test: echo 'hello' | pbcopy (requires terminal with OSC 52 support)"
else
    echo "❌ pbcopy installation failed"
    exit 1
fi
```

## 4. Usage Examples

- **opencode**: After executing `source ~/.profile`, directly run `opencode` to start the AI programming assistant.
- **pbcopy**: `echo "hello" | pbcopy`, copy content to local clipboard (requires terminal to support OSC 52).

## 5. Follow-up Recommendations

- **Configure opencode**: Refer to opencode documentation to set up model API Key and configuration files (`~/.config/opencode/`).
- **Configure PostgreSQL environment variables**: Refer to `/data/service/pg-unires/README.md` to set `PGUSER`, `PGPASSWORD`, etc.
- **Download model files**: Execute `/scripts/download_models.py` to pull Qwen quantized models to `/data/models`.

---

> ✅ At this point, application tool deployment is complete, you can continue deploying other Uni-Resource Agent components.